In [0]:
%sql
CREATE SCHEMA IF NOT EXISTS main.silver;

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

try:
    # Leitura da tabela da camada Bronze
    df_bronze_customers = spark.table("main.bronze.tb_customers")

    # Mapeamento e Renomeação de Colunas
    df_renomeado = df_bronze_customers \
        .withColumnRenamed("customer_id", "id_consumidor") \
        .withColumnRenamed("customer_unique_id", "id_unico_consumidor") \
        .withColumnRenamed("customer_zip_code_prefix", "prefixo_cep") \
        .withColumnRenamed("customer_name", "nome_consumidor") \
        .withColumnRenamed("customer_city", "cidade") \
        .withColumnRenamed("customer_state", "estado") \
        .withColumnRenamed("customer_gender", "genero") \
        .withColumnRenamed("customer_birth_date", "data_nascimento") \
        .withColumnRenamed("timestamp_ingestion", "data_ingestao") \
        .withColumnRenamed("customer_age", "idade")

    # Transformações: Upper Case para Cidade e Estado
    df_transformado = df_renomeado \
        .withColumn("cidade", F.upper(F.col("cidade"))) \
        .withColumn("estado", F.upper(F.col("estado"))) \
        .withColumn("prefixo_cep", F.col("prefixo_cep").cast("string"))

    # Deduplicação Sênior com Window Functions
    window_spec = Window.partitionBy("id_consumidor").orderBy(F.col("data_ingestao").desc())

    df_deduplicado = df_transformado \
        .withColumn("row_num", F.row_number().over(window_spec)) \
        .filter(F.col("row_num") == 1) \
        .drop("row_num") # Removemos a coluna auxiliar, pois já não é necessária

    # Guardar na camada Silver como tabela Delta (dim_consumidores)
    destino = "main.silver.dim_consumidores"
    df_deduplicado.write \
        .format("delta") \
        .mode("overwrite") \
        .option("overwriteSchema", "true") \
        .saveAsTable(destino)

    print(f"Sucesso: Tabela {destino} criada e deduplicada com sucesso!")

except Exception as e:
    print(f"Erro ao processar dim_consumidores: {e}")

In [0]:
display(spark.table("main.silver.dim_consumidores"))

In [0]:
from pyspark.sql import functions as F

print("Processando a tabela: fat_pedidos")

try:
    # Leitura da tabela da camada Bronze
    df_bronze_orders = spark.table("main.bronze.tb_orders")

    # Tradução dos nomes das colunas originais para o português
    df_renomeado = df_bronze_orders \
        .withColumnRenamed("order_id", "id_pedido") \
        .withColumnRenamed("customer_id", "id_consumidor") \
        .withColumnRenamed("order_status", "status_pedido") \
        .withColumnRenamed("order_purchase_timestamp", "data_compra") \
        .withColumnRenamed("order_approved_at", "data_aprovacao") \
        .withColumnRenamed("order_delivered_carrier_date", "data_envio_transportadora") \
        .withColumnRenamed("order_delivered_customer_date", "data_entrega_cliente") \
        .withColumnRenamed("order_estimated_delivery_date", "data_estimada_entrega") \
        .withColumnRenamed("timestamp_ingestion", "data_ingestao")

    # Mapeamento dos valores da coluna de status
    df_status = df_renomeado.withColumn(
        "status_pedido",
        F.when(F.col("status_pedido") == "delivered", "entregue")
         .when(F.col("status_pedido") == "canceled", "cancelado")
         .when(F.col("status_pedido") == "shipped", "enviado")
         .when(F.col("status_pedido") == "processing", "processando")
         .when(F.col("status_pedido") == "invoiced", "faturado")
         .when(F.col("status_pedido") == "unavailable", "indisponível")
         .when(F.col("status_pedido") == "created", "criado")
         .when(F.col("status_pedido") == "approved", "aprovado")
         .otherwise(F.col("status_pedido"))
    )

    # Criação das novas colunas derivadas de datas
    df_metricas = df_status \
        .withColumn("tempo_entrega_dias", F.datediff(F.col("data_entrega_cliente"), F.col("data_compra"))) \
        .withColumn("tempo_entrega_estimado_dias", F.datediff(F.col("data_estimada_entrega"), F.col("data_compra"))) \
        .withColumn("diferenca_entrega_dias", F.col("tempo_entrega_dias") - F.col("tempo_entrega_estimado_dias"))

    # Criação do indicador de entrega no prazo
    # Se o status não for 'entregue', fica 'Não Entregue'.
    # Se a diferença for <= 0 (chegou antes ou no dia exato do prazo), é 'Sim'. Caso contrario, 'Não'.
    df_final = df_metricas.withColumn(
        "entrega_no_prazo",
        F.when(F.col("status_pedido") != "entregue", "Não Entregue")
         .when(F.col("diferenca_entrega_dias") <= 0, "Sim")
         .otherwise("Não")
    )

    # Salvar como tabela Delta
    destino = "main.silver.fat_pedidos"
    df_final.write \
        .format("delta") \
        .mode("overwrite") \
        .option("overwriteSchema", "true") \
        .saveAsTable(destino)

    print(f"Sucesso: Tabela {destino} gerada perfeitamente com todas as regras!")

except Exception as e:
    print(f"Erro ao processar fat_pedidos: {e}")

In [0]:
display(spark.table("main.silver.fat_pedidos"))

In [0]:
from pyspark.sql import functions as F

print("Processando a tabela: fat_itens_pedidos")

try:
    # Leitura da tabela da camada Bronze
    df_bronze_order_items = spark.table("main.bronze.tb_order_items")

    # Mapeamento e Renomeação de Colunas
    df_renomeado = df_bronze_order_items \
        .withColumnRenamed("order_id", "id_pedido") \
        .withColumnRenamed("order_item_id", "id_item") \
        .withColumnRenamed("product_id", "id_produto") \
        .withColumnRenamed("seller_id", "id_vendedor") \
        .withColumnRenamed("shipping_limit_date", "data_limite_envio") \
        .withColumnRenamed("price", "preco_BRL") \
        .withColumnRenamed("freight_value", "preco_frete") \
        .withColumnRenamed("timestamp_ingestion", "data_ingestao")

    # Guardar na camada Silver como tabela Delta
    destino = "main.silver.fat_itens_pedidos"
    df_renomeado.write \
        .format("delta") \
        .mode("overwrite") \
        .option("overwriteSchema", "true") \
        .saveAsTable(destino)

    print(f"Sucesso: Tabela {destino} criada com todas as colunas padronizadas!")

except Exception as e:
    print(f"Erro ao processar fat_itens_pedidos: {e}")

In [0]:
from pyspark.sql import functions as F

print("Processando a tabela: fat_pagamentos_pedidos")

try:
    # Leitura da tabela da camada Bronze
    df_bronze_payments = spark.table("main.bronze.tb_order_payments")

    # Renomeação de todas as colunas para português
    df_renomeado = df_bronze_payments \
        .withColumnRenamed("order_id", "id_pedido") \
        .withColumnRenamed("payment_sequential", "sequencial_pagamento") \
        .withColumnRenamed("payment_type", "tipo_pagamento") \
        .withColumnRenamed("payment_installments", "parcelas") \
        .withColumnRenamed("payment_value", "valor_pagamento") \
        .withColumnRenamed("timestamp_ingestion", "data_ingestao")

    # Mapeamento dos tipos de pagamento (Tradução dos valores da coluna)
    df_transformado = df_renomeado.withColumn(
        "tipo_pagamento",
        F.when(F.col("tipo_pagamento") == "credit_card", "Cartão de Crédito")
         .when(F.col("tipo_pagamento") == "boleto", "Boleto")
         .when(F.col("tipo_pagamento") == "voucher", "Voucher")
         .when(F.col("tipo_pagamento") == "debit_card", "Cartão de Débito")
         .when(F.col("tipo_pagamento") == "not_defined", "Não Definido")
         .otherwise(F.col("tipo_pagamento")) # Mantém original caso surja um tipo novo no futuro
    )

    # Guardar na camada Silver como tabela Delta
    destino = "main.silver.fat_pagamentos_pedidos"
    df_transformado.write \
        .format("delta") \
        .mode("overwrite") \
        .option("overwriteSchema", "true") \
        .saveAsTable(destino)

    print(f"Sucesso: Tabela {destino} criada com pagamentos traduzidos!")

except Exception as e:
    print(f"Erro ao processar fat_pagamentos_pedidos: {e}")

In [0]:
fat_pagamentos_pedidos = spark.table("main.silver.fat_pagamentos_pedidos")
fat_pagamentos_pedidos.display()